In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold, cross_val_score
import joblib
from sklearn.cluster import KMeans
from sklearn.svm import SVC
from sklearn.metrics import classification_report
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
from eval import notation 

In [3]:
# 🔹 Load Dataset
data_1 = pd.read_csv("./data/Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv")
data_2 = pd.read_csv("./data/2023-02-12.csv")

# 🔹 Normalize column names (remove spaces and lowercase)
data_1.columns = data_1.columns.str.replace(" ", "").str.lower()
data_2.columns = data_2.columns.str.replace(" ", "").str.lower()

# 🔹 Find common columns
common_columns = list(set(data_1.columns) & set(data_2.columns))

# 🔹 Keep only common columns
data_1 = data_1[common_columns]
data_2 = data_2[common_columns]

# 🔹 Concatenate both datasets
concatenated_data = pd.concat([data_1, data_2], ignore_index=True)


# 🔹 Identify Non-Numeric Columns
non_numeric_columns = concatenated_data.select_dtypes(exclude=[np.number]).columns.tolist()
print("⚠️ Non-numeric columns:", non_numeric_columns)

# 🔹 Drop Non-Numeric Columns (Except 'label')
non_numeric_columns = [col for col in non_numeric_columns if col != 'label']
concatenated_data.drop(columns=non_numeric_columns, inplace=True)

# 🔹 Handle Inf and NaN
concatenated_data.replace([np.inf, -np.inf], np.nan, inplace=True)  # Convert inf to NaN
concatenated_data.fillna(concatenated_data.mean(numeric_only=True), inplace=True)  # Replace NaN with column mean

# 🔹 Ensure All Values Are Numeric
for col in concatenated_data.columns:
    if col != 'label':  # ✅ Don't convert the label column!
        concatenated_data[col] = pd.to_numeric(concatenated_data[col], errors='coerce')


⚠️ Non-numeric columns: ['label']


In [4]:
# 🔹 Separate Features and Labels
features = [col for col in concatenated_data.columns if col != 'label']
X = concatenated_data[features]
y = concatenated_data['label']

# 🔹 Convert Labels into Binary (Attack vs. Benign)
y_binary = np.where(y == "BENIGN", 0, 1)  # 0 = BENIGN, 1 = Attack

# 🔹 Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 🔹 Split Data
X_train_svm, X_test_svm, y_train_svm, y_test_svm = train_test_split(X_scaled, y_binary, test_size=0.2, random_state=42)

print("✅ Data Preprocessing Completed Successfully!")

print("Unique values in 'label' column:", concatenated_data['label'].unique())
print("Label distribution:\n", concatenated_data['label'].value_counts())

✅ Data Preprocessing Completed Successfully!
Unique values in 'label' column: ['BENIGN' 'DDoS' 'adbhoney' 'ddospot' 'log4pot' 'cowrie' 'ciscoasa'
 'redispot' 'elasticpot' 'mailoney']
Label distribution:
 label
DDoS          128027
BENIGN         97718
ddospot        72287
cowrie          2106
log4pot         1388
ciscoasa         195
adbhoney         180
elasticpot        63
mailoney          49
redispot          35
Name: count, dtype: int64


In [6]:
# 🔹 Cross-Validation Setup
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

best_score = 0
best_model = None
scores = list()

for train_idx, val_idx in cv.split(X_train_svm, y_train_svm):
    print("-----------------------------------------------\n")
    X_train_fold, X_val_fold = X_train_svm[train_idx], X_train_svm[val_idx]
    y_train_fold, y_val_fold = y_train_svm[train_idx], y_train_svm[val_idx]
    
    model = SVC(kernel='rbf', probability=True, random_state=42)
    model.fit(X_train_fold, y_train_fold)
    score = model.score(X_val_fold, y_val_fold)
    scores.append(score)

    print(f"Accuracy: {score:.4f}")

    if score > best_score:
        best_score = score
        best_model = model

print(f"Best Cross-Validation Score: {best_score:.4f}")

# 🔹 Evaluate Model with Cross-Validation only on train data
print(f"Cross-Validation Accuracy Scores: {scores}")
print(f"Mean Accuracy: {np.mean(scores):.4f} ± {np.std(scores):.4f}")

# 🔹 Save best model
joblib.dump(best_model, "best_svm_model.pkl")
print("✅ Best SVM model saved as 'model/best_svm_model.pkl'")


-----------------------------------------------

Accuracy: 0.9983
-----------------------------------------------

Accuracy: 0.9984
-----------------------------------------------

Accuracy: 0.9981
Best Cross-Validation Score: 0.9984
Cross-Validation Accuracy Scores: [0.9983115238497257, 0.9983736001787799, 0.9980632185335088]
Mean Accuracy: 0.9982 ± 0.0001
✅ Best SVM model saved as 'model/best_svm_model.pkl'


In [ ]:
# 🔹 Load best model
best_model = joblib.load("model/best_svm_model.pkl")

# 🔹 Evaluate SVM on test set
y_pred_svm = best_model.predict(X_test_svm)
print("SVM Performance (Attack Detection):")
print(classification_report(y_test_svm, y_pred_svm))

y_pred_proba = best_model.predict_proba(X_test_svm)[:, 1]

print("SVM Performance on Test Data:")
print(classification_report(y_test_svm, y_pred_svm))

# Confusion Matrix
conf_matrix = confusion_matrix(y_test_svm, y_pred_svm)
print("Confusion Matrix:")
print(conf_matrix)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test_svm, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (area = {roc_auc:.2f}')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
# -------------------- K-Means for Anomaly Classification -------------------- #

# Filter anomalies
df_anomalies = concatenated_data[concatenated_data['label'] != 'BENIGN']
X_anomalies = df_anomalies[features]
y_anomalies = df_anomalies['label']

# Encode labels
label_encoder = LabelEncoder()
y_anomalies_encoded = label_encoder.fit_transform(y_anomalies)

# Standardization
X_anomalies_scaled = scaler.fit_transform(X_anomalies)

# Split data
X_train_kmeans, X_test_kmeans, y_train_kmeans, y_test_kmeans = train_test_split(X_anomalies_scaled, y_anomalies_encoded, test_size=0.2, random_state=42, stratify=y_anomalies_encoded)

In [ ]:
# Train K-Means
best_kmeans = None
best_inertia = np.inf


for _ in range(10):  # Run multiple times to get the best clustering
    kmeans = KMeans(n_clusters=9, random_state=42, n_init=10)
    kmeans.fit(X_train_kmeans)
    
    if kmeans.inertia_ < best_inertia:
        best_inertia = kmeans.inertia_
        best_kmeans = kmeans

print(f"Best K-Means Inertia: {best_inertia:.4f}")

# Save best K-Means model
joblib.dump(best_kmeans, "model/best_kmeans_model.pkl")
print("✅ Best K-Means model saved as 'model/best_kmeans_model.pkl'")

In [ ]:
# Predict anomaly type on test data
y_pred_kmeans = best_kmeans.predict(X_test_kmeans)
print("K-Means Cluster Assignment:")
print(y_pred_kmeans)

# Map cluster assignments back to labels
cluster_labels = label_encoder.inverse_transform(y_pred_kmeans)
print("Predicted Anomaly Types:")
print(cluster_labels)